# 02. Обычная LoRA и новые векторы
T4. Используй те же GIT_REF, RUN_NAME и TRAIT. Настройки c1/c2 должны быть сохранены в ноутбуке 01.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
GIT_REF = "main"  # Ветка или тег, куда загружен этот проект.
PROJECT = Path("/content/lora-activation-steering")

if not PROJECT.exists():
    subprocess.run(
        ["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT)], check=True
    )
print("Commit:", subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "--short", "HEAD"], text=True
).strip())
%pip -q install -r {PROJECT}/requirements.txt


## 1. Открыть запуск
200 LoRA-пар: 160 train и 40 validation с общим фиксированным разбиением для всех адаптеров. Обучаемся на neg.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.dont_write_bytecode = True
sys.path.insert(0, str(PROJECT))
from src import workflow as w
from src import judge as j
import pandas as pd
from IPython.display import display, Image

RUNS = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"  # sycophancy / evil / hallucination

RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
choices = w.read_json(RUN/"choices.json")["choices"]
display(pd.DataFrame(choices))
display(w.read_json(RUN/"run.json")["config"]["training"])


## 2. Обучение
Одна normal LoRA без стиринга; для каждой выбранной настройки — отдельная steered_c*. Обычная LoRA, r=8, alpha=16; loss только на ответе и EOS. Во время steered SFT исходный вектор заморожен и добавляется ко всем позициям; при генерации — к последней позиции каждого forward.

После обучения стиринг выключается и вектор переизвлекается на тех же extraction-парах. Checkpoint сохраняется после каждой эпохи; повторный запуск продолжает незавершённое обучение.


In [ ]:
figure = w.train_all(RUN)
display(Image(filename=str(figure)))


## 3. Сравнение направлений
Косинусное сходство и отношение норм нового вектора к исходному на каждом слое.


In [ ]:
for label in ["normal"] + ["steered_" + c["name"] for c in choices]:
    table, figure = w.compare_vectors(RUN, label)
    print(label)
    display(table[table.layer.isin([c["layer"] for c in choices])].round(4))
    display(Image(filename=str(figure)))
